In [ ]:
import xarray as xr
import numpy as np

# For retrieving the time at which the dataset was modified
from datetime import datetime, timedelta, UTC

ds_ss=xr.open_dataset('../ds_ss.nc') # NOTE: find the file in the description

def get_ds(like_variable, data,
           lat_resolution = 0,
           lon_resolution = 0,
           time_resolution = 0):
    
    """ Function to create a netCDF/CF and ACDD compliant dataset using xarray, such that results can be written to disk. 

        Args:
            like_variable (xr.Dataset/xr.DataArray): An array_like-variable for creating an output variable. Will contain longitude/latiude data etc.
            data (np.ndarray): array of data for populating the like_variable
        Kwargs:
            lat_resolution: The resolution of the latitude coordinate in the dataset. If zero, infers value assuming a regular lat/lon grid. If None, set to N/A 
            lon_resolution: The resolution of the longitude coordinate in the dataset. If zero, infers value assuming a regular lat/lon grid. If None, set to N/A 
            time_resolution: The resolution of the time coordinate in the dataset. If zero, infers value based on axis. If None, set to N/A 

        Returns:
            ds (xr.Dataset/xr.DataArray): A netCDF/CF and ACDD compliant dataset

    
    """

    # Define some helperfunctions

    def td_to_iso(td):
        # Computes np.timedelta64 to ISO8601 standard
        # cannot handle variable-length units such as months and years
        
        nanoseconds = td.astype("timedelta64[ns]").astype(int)
        
        minutes, seconds = divmod(nanoseconds, 60e9)
        hours, minutes = divmod(minutes, 60)
        days, hours = divmod(hours, 24)

        # convert microseconds to fractional seconds
        microseconds = str(td.microseconds/1e6)[1:]

        return f"P{days}D{hours}H{minutes}M{seconds}S"
    
    def maxdelta(ds, coord):
        # computes the maximum delta for a dataset and a specified coordinate
        return np.max(abs(ds[coord][1:] - ds[coord][:-1]).values)


    ds = xr.zeros_like(like_variable)
    ds.name='zeta' # Vertical component of the relative vorticity vector (ie., \nabla \dot U_vector)
    ds.attrs["units"] = "s-1" # 
    ds.attrs["description"] = "Estimated vertical vorticity component from CoastCurr"
    ds = ds.rename({"latitude" : "lat",
                    "longitude" : "lon"})


    # ================================================================================================
    # HERE WE COMPUTE DATASET-SPECIFIC ATTRIBUTES

    # compute the spatial extent
    lat_min = ds.coords["lat"].min().item()
    lat_max = ds.coords["lat"].max().item()

    lon_min = ds.coords["lon"].min().item()
    lon_max = ds.coords["lon"].max().item()

    # compute spatial resolution, assuming regular grid
    if lat_resolution == 0:
        lat_resolution = maxdelta(ds, "lat")
    if lon_resolution == 0:
        lon_resolution = maxdelta(ds, "lon")

    # compute date_modified
    date_modified = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")

    # compute start and endtime

    # if not given, compute the temporal resolution based on the maximum differences 
    if (time_resolution == 0) and ("time" in data.coords):
        try:
            time_ordered = np.sort(data.coords["time"].values)
            time_ordered_max = np.max(abs(time_ordered[1:]-time_ordered[:-1]))

            # check two different formats of time:
            if isinstance(time_resolution, np.timedelta64):
                # if time_ordered_max is in timedelta form:
                time_resolution = td_to_iso(time_ordered_max)
            elif isinstance(time_resolution, np.float64):
                # if time passed as floats, we assume seconds:
                time_resolution = td_to_iso(np.timedelta64(time_ordered_max, "s"))
            else:
                raise TypeError
                
        except TypeError:
            print("Unknown time format, time_coverage_resolution could not be inferred")
            time_resolution = "N/A"
    else:
        time_resolution = "N/A"


    # ================================================================================================


    #... attributes (both global and variable) making it CF and ACDD-compliant
    # We orient ourselves along https://wiki.esipfed.org/Attribute_Convention_for_Data_Discovery_1-3

    attrs = {
        # HIGHLY RECOMMENDED:
        "title" : "N/A",
        "summary" : "N/A",
        "keywords" : "N/A",
        "Conventions" : "CF-1.6, ACDD-1.3",

        # RECOMMENDED:
        "id" : "N/A",
        "naming_authority" : "N/A",
        "history" : "N/A",
        "source" : "N/A",
        "processing_level" : "N/A",
        "comment" : "N/A",
        "acknowledgement" : "N/A",
        "license" : "N/A",
        "standard_name_vocabulary" : "N/A",
        "date_created" : "N/A",
        "creator_name" : "N/A",
        "creator_email" : "N/A",
        "creator_url" : "N/A",
        "institution" : "N/A",
        "project" : "N/A",
        "publisher_name" : "N/A",
        "publisher_email" : "N/A",
        "geospatial_bounds" : "N/A",
        "geospatial_bounds_crs" : "N/A",
        "geospatial_lat_min" : lat_min,
        "geospatial_lat_max" : lat_max,
        "geospatial_lon_min" : lon_min,
        "geospatial_lon_max" : lon_max,
        "time_coverage_start" : "N/A",
        "time_coverage_end" : "N/A",
        "time_coverage_duration" : "N/A",
        "time_coverage_resolution" : time_resolution,
        
        # SUGGESTED:
        "creator_type" : "N/A",
        "creator_institution" : "N/A",
        "publisher_type" : "N/A",
        "publisher_institution" : "N/A",
        "program" : "N/A",
        "geospatial_lat_units" : "N/A",
        "geospatial_lat_resolution" : "N/A",
        "geospatial_lon_units" : "N/A",
        "geospatial_lon_resolution" : "N/A",
        "date_modified" : date_modified,
    }

    ds.attrs = attrs

    ds[:]=data

    return ds

ds_ss

<xarray.Dataset> Size: 52MB
Dimensions:                                (num_lines: 2600, num_pixels: 173)
Coordinates:
    latitude                               (num_lines, num_pixels) float64 4MB ...
    longitude                              (num_lines, num_pixels) float64 4MB ...
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/20)
    time                                   (num_lines) datetime64[ns] 21kB ...
    time_tai                               (num_lines) datetime64[ns] 21kB ...
    latitude_uncert                        (num_lines, num_pixels) float64 4MB ...
    longitude_uncert                       (num_lines, num_pixels) float64 4MB ...
    polarization_karin                     (num_lines) object 21kB ...
    ssh_karin_2                            (num_lines, num_pixels) float64 4MB ...
    ...                                     ...
    miti_power_var_250m                    (num_lines, num_pixels) float32 2MB ...
    ancillary_surface_classification_flag  (num_lines, num_pixels) float32 2MB ...
    bathy                                  (num_lines, num_pixels) float64 4MB ...
    eta_detrend                            (num_lines, num_pixels) float64 4MB ...
    x                                      (num_pixels) float64 1kB ...
    y                                      (num_lines) float64 21kB ...
Attributes:
    description:  Unsmoothed SSH measurement data and related information for...

In [33]:
ds_gridded_zeta = get_ds(like_variable=ds_ss.eta_detrend,
                        data = np.ones(ds_ss.eta_detrend.shape),
                        )

ds_gridded_zeta

#ds_gridded_zeta.to_netcdf('../data/my_CF-ACDD-compliant_netCDF-file.nc')

<xarray.DataArray 'zeta' (num_lines: 2600, num_pixels: 173)> Size: 4MB
array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]], shape=(2600, 173))
Coordinates:
    lat      (num_lines, num_pixels) float64 4MB 59.72 59.72 ... 54.67 54.67
    lon      (num_lines, num_pixels) float64 4MB 215.2 215.2 ... 219.5 219.5
Dimensions without coordinates: num_lines, num_pixels
Attributes: (12/41)
    title:                      N/A
    summary:                    N/A
    keywords:                   N/A
    Conventions:                CF-1.6, ACDD-1.3
    id:                         N/A
    naming_authority:           N/A
    ...                         ...
    program:                    N/A
    geospatial_lat_units:       N/A
    geospatial_lat_resolution:  N/A
    geospatial_lon_units:       N/A
    geospatial_lon_resolution:  N/A
    date_modified:              20260629T115058Z